In [1]:
import ee
import geemap
import math 
import os

In [2]:


# 设置代理（替换成你的代理地址）
os.environ["HTTP_PROXY"] = "http://127.0.0.1:7897"
os.environ["HTTPS_PROXY"] = "http://127.0.0.1:7897"

In [4]:

ee.Authenticate()
ee.Initialize()



Successfully saved authorization token.


In [5]:
Map = geemap.Map()

In [6]:
print('Hello world')

Hello world


In [7]:
ee.Initialize()

In [8]:
from utils.utils import *
from utils.ee_utils import *
from utils.utils import TextColors as c

In [22]:
# df = read_csv("LUCAS_2015_all.csv")
df = pd.read_csv('China-SOCD-Dataset-20250415_filter0-30_resorted.CSV')
df = df.iloc[1190:]
df 


,Point_id,SOCD (kg/m2),Latitude,Longitude,Upper_depth (cm),Lower_depth (cm)
1190,1191,0.72,35.72,79.37,0,30
1191,1192,0.55,35.77,79.41,0,30
1192,1193,1.39,35.77,79.39,0,30
1193,1194,0.76,35.81,78.95,0,30
1194,1195,0.96,35.82,79.41,0,30
...,...,...,...,...,...,...
1337,1338,3.45,31.52,93.67,0,30
1338,1339,6.78,31.53,93.41,0,30
1339,1340,3.84,31.52,93.18,0,30
1340,1341,3.04,31.70,93.17,0,30


In [17]:
DATASET = 'CN' #'LUCAS'
if DATASET == 'CN':
 create_folder_if_not_exists('l8_images_CN')
 DFOLDER = 'l8_images_CN//'
if DATASET == 'RaCA':
 create_folder_if_not_exists('l8_images_us')
 DFOLDER = 'l8_images_us//'

if DATASET == 'LUCAS':
 create_folder_if_not_exists('l8_images')
 DFOLDER = 'l8_images//'

Folder "l8_images_CN" already exists in the current working directory.


### Ignore the following cell, some kind of data cleaning

In [9]:
# # Step 1: Get the list of TIFF file names
# tif_folder_path = "C:\\Users\\nkakhani\\_Multimodal\\SoilNet-7\\SoilNet-PreRelease\\dataset\\l8_images_us"
# tif_files = [f for f in os.listdir(tif_folder_path) if f.endswith('.tif')]

# # Extract the first part of each TIFF file name
# tiff_first_parts = [tif.split('_')[0] for tif in tif_files]
# print(len(tiff_first_parts))

# # Step 3: Find Point IDs in the CSV DataFrame that are not in tiff_first_parts
# missing_point_ids = df[~df['Point_ID'].isin(tiff_first_parts)]['Point_ID']

# # Print or use missing_point_ids as needed
# print("Point IDs in CSV DataFrame not present in TIFF file names:")
# print(missing_point_ids)

# # # Step 3: Check if each first part is present in the specific column of the CSV file
# # for tiff_first_part, tif_file in zip(tiff_first_parts, tif_files):
# #     if tiff_first_part not in df['Point_ID'].values:
# #         # Step 4: If not present, delete the file immediately
# #         file_path = os.path.join(tif_folder_path, tif_file)
# #         os.remove(file_path)
# #         print(f"Deleted {tif_file}")

In [18]:
# Config
IMAGE_COLLECTION = ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')
WINTER_RANGES = [('2015-01-01', '2015-02-28'), ('2015-11-01', '2015-12-31')]
NONE_WINTER_RANGES = (WINTER_RANGES[0][1], WINTER_RANGES[1][0])
#Required Landsat bands and RS indices
BANDS = ['SR_B1','SR_B2','SR_B3','SR_B4','SR_B5','SR_B6','SR_B7','clayIndex',
         'ferrousIndex','carbonateIndex','rockOutcropIndex','NDVI','elevation','slopePCT']

In [23]:
winter_ragne_1 = ee.Filter.date(WINTER_RANGES[0][0], WINTER_RANGES[0][1])
winter_range_2 = ee.Filter.date(WINTER_RANGES[1][0], WINTER_RANGES[1][1])
winter_range_filter = ee.Filter.Or(winter_ragne_1, winter_range_2)

non_winter_range_filter = ee.Filter.date(NONE_WINTER_RANGES[0], NONE_WINTER_RANGES[1])

for index, row in df.iterrows():
    # print(f'{c.BOLD} Point {int(row["Piont_id"])} {c.ENDC}' , end=" Cheking for Winter Image -> ")
    print(f'{c.BOLD} Point {(row["Point_id"])} {c.ENDC}' , end=" Cheking for Winter Image -> ")
    loop_roi = get_square_roi(row['Latitude'],row['Longitude'], roi_size=1920, return_gee_object=True)
    
    # FIRST: Checkig if we can find an image in the WINTER_RANGE
    l8 = IMAGE_COLLECTION.filter(winter_range_filter)\
                         .filterBounds(loop_roi).sort('system:time_start')
    if l8.size().getInfo() > 0:                     
        #clip the image collection to the roi                     
        l8 = l8.map(lambda img: img.clip(loop_roi))
        topoBands = add_topo().clip(loop_roi)
        
        # discard images with high null pixels
        l8 = l8.map(lambda img: img.set('not_null_pixels', get_not_nulls_ratio(img,loop_roi)))
        l8 = l8.filter(ee.Filter.gt('not_null_pixels',0.7))
        
        # Ratiometric correction
        l8 = l8.map(lambda img: radiometric_correction(img))
        # add roi cloud cover and cloud shadow property # the index 2 is for the combination of cloud and cloud shadow mask
        l8 = l8.map(lambda img: img.set('roi_cloud_cover', get_mask_ones_ratio(get_cloud_mask(img)[2])))
        l8 = l8.filter(ee.Filter.lt('roi_cloud_cover',10)) 
        
        # add roi snow cover property
        l8 = l8.map(lambda img: img.set('roi_snow_cover', get_mask_ones_ratio(get_snow_mask(img))))
        l8 = l8.filter(ee.Filter.lt('roi_snow_cover',10)) 

    if l8.size().getInfo() > 0:
        print (f'{c.OKGREEN} Found {c.ENDC}')
        l8_image = add_mineral_indices(l8.sort('system:time_start').first()).addBands(topoBands)
        # print('Band names:', l8_image.bandNames().getInfo())

    else: # if we can't find an image in the WINTER_RANGE, we check the NONE_WINTER_RANGE
        print (f'{c.FAIL} Not Found {c.ENDC}', end=" Cheking for None Winter Image -> ")
        l8 = IMAGE_COLLECTION.filter(non_winter_range_filter)\
                             .filterBounds(loop_roi).sort('system:time_start')
                             
        #clip the image collection to the roi                     
        l8 = l8.map(lambda img: img.clip(loop_roi))
                    
        # discard images with high null pixels
        l8 = l8.map(lambda img: img.set('not_null_pixels', get_not_nulls_ratio(img,loop_roi)))
        l8 = l8.filter(ee.Filter.gt('not_null_pixels',0.7))

        # Ratiometric correction
        l8 = l8.map(lambda img: radiometric_correction(img))
        
        # add roi cloud cover and cloud shadow property # the index 2 is for the combination of cloud and cloud shadow mask
        l8 = l8.map(lambda img: img.set('roi_cloud_cover', get_mask_ones_ratio(get_cloud_mask(img)[2])))
        l8 = l8.filter(ee.Filter.lt('roi_cloud_cover',10)) 
        
        # add roi snow cover property
        l8 = l8.map(lambda img: img.set('roi_snow_cover', get_mask_ones_ratio(get_snow_mask(img))))
        l8 = l8.filter(ee.Filter.lt('roi_snow_cover',10)) 
        
        l8 = l8.map(lambda img: img.set('roi_mean_ndvi', get_mean_ndvi(img)))  
        
        if l8.size().getInfo() > 0:
            print(f'{c.OKGREEN} Found {c.ENDC}')
            l8_image = add_mineral_indices(l8.sort('roi_mean_ndvi').first()).addBands(topoBands)
            # print('Band names:', l8_image.bandNames().getInfo())
        else:
            print(f'{c.FAIL} No Image Found For this ROI! {c.ENDC}')

    if l8.size().getInfo() > 0:    
        date = ee.Date(l8_image.get('system:time_start')).format('YYMMdd').getInfo()
        # name = f"{int(row['Piont_id'])}_{date}.tif"
        name = f"{(row['Point_id'])}_{date}.tif"
        geemap.download_ee_image(l8_image.select(BANDS), DFOLDER+ name,crs='EPSG:3857', scale=30, region = loop_roi)
                       
    #print('Date:',milsec2date(l8.aggregate_array('system:time_start').getInfo()))
    
    # if index == 20:
    #     break

 Point 1191.0  Cheking for Winter Image ->  Not Found  Cheking for None Winter Image ->  Found 


1191.0_150706.tif: |          | 0.00/708k (raw) [  0.0%] in 00:00 (eta:     ?)

 Point 1192.0  Cheking for Winter Image ->  Found 


1192.0_150111.tif: |          | 0.00/717k (raw) [  0.0%] in 00:00 (eta:     ?)

 Point 1193.0  Cheking for Winter Image ->  Found 


...

1340.0_151112.tif: |          | 0.00/638k (raw) [  0.0%] in 00:00 (eta:     ?)

 Point 1341.0  Cheking for Winter Image ->  Found 


1341.0_151121.tif: |          | 0.00/647k (raw) [  0.0%] in 00:00 (eta:     ?)

 Point 1342.0  Cheking for Winter Image ->  Found 


1342.0_151214.tif: |          | 0.00/647k (raw) [  0.0%] in 00:00 (eta:     ?)

In [ ]:
Map = geemap.Map()

In [ ]:
sdf = (123,)
len(sdf)

## Testing some outputs

In [ ]:
date = ee.Date(l8_image.get('system:time_start')).format('YYMMdd').getInfo()
date

In [ ]:
Map = geemap.Map()
# l8_c_img = l8.sort('roi_mean_ndvi',True).first()
l8_c_img = l8.sort('system:time_start',True).first()
print(l8_c_img.get('roi_mean_ndvi').getInfo())
visualization = {
  'bands': ['SR_B4', 'SR_B3', 'SR_B2'],
  'min': 0.0,
  'max': 0.3,
}

cloud = get_cloud_mask(l8_c_img)[2]


Map.addLayer(cloud, {'min': 0, 'max': 1, 'palette': ['black','red']}, 'cloud')
Map.addLayer(l8_c_img, visualization, 'rgb')
Map.centerObject(loop_roi)
Map